# Guía de Elementos y Conceptos — Práctica 3: Segmentación, Sustracción de Fondo y Template Matching

Esta guía resume las **funciones, parámetros, interpretaciones teóricas y plantillas de código** trabajadas en la Práctica 3, estructuradas por bloques temáticos para consulta rápida en el examen.

### Índice de Bloques
1. **[Resumen Rápido de Funciones de la Práctica 3](#1-tabla-funciones)**
2. **[Segmentación por Superpíxeles y Grafos: Felzenszwalb, SLIC y Quickshift](#2-segmentacion-skimage)**
3. **[Segmentación mediante Clustering Jerárquico (`AgglomerativeClustering`)](#3-clustering-jerarquico)**
4. **[Sustracción de Fondo para Detección de Movimiento (*Background Subtraction*)](#4-background-subtraction)**
5. **[Template Matching: SSD vs Correlación Cruzada Normalizada (NCC)](#5-template-matching)**
6. **[Invarianza a la Rotación en Template Matching (`transform.rotate`)](#6-rotacion-template)**

In [ ]:
# Imports estándar utilizados en toda la Práctica 3
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from skimage import io, color, transform, filters, util, feature, measure, segmentation, img_as_float
from skimage.segmentation import felzenszwalb, slic, quickshift, mark_boundaries
from skimage.transform import resize, rotate
from skimage.feature import match_template

from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph

<a id='1-tabla-funciones'></a>
---
## 1. Resumen Rápido de Funciones de la Práctica 3

| Tarea | Función / Expresión | Parámetros Clave y Comportamiento |
| :--- | :--- | :--- |
| **Segmentación Felzenszwalb** | `segmentation.felzenszwalb(img, scale=100, sigma=0.5, min_size=50)` | Basada en grafos. `scale` controla tamaño de regiones; `sigma` suaviza antes; `min_size` fusiona regiones pequeñas |
| **Segmentación SLIC** | `segmentation.slic(img, n_segments=250, compactness=10, sigma=1, start_label=1)` | Basada en K-means (color + espacio). `n_segments` fija nº aprox. de superpíxeles; `compactness` regula cuán cuadrados son |
| **Segmentación Quickshift** | `segmentation.quickshift(img, kernel_size=3, max_dist=6, ratio=0.5)` | Basada en densidad. Mayor `kernel_size` o `max_dist` = menos segmentos (más grandes); `ratio` balancea color vs espacio |
| **Dibujar bordes de segmentos** | `segmentation.mark_boundaries(img, segments)` | Superpone los contornos de la matriz de etiquetas `segments` sobre `img` |
| **Contar nº de segmentos** | `len(np.unique(segments))` | Cuenta cuántas etiquetas únicas hay en el mapa de segmentación |
| **Añadir ruido Sal y Pimienta** | `util.random_noise(img, mode='s&p', amount=0.05)` | Devuelve imagen `float64` en `[0, 1]` con un 5% de píxeles blancos/negros |
| **Suavizado Gaussiano en RGB** | `filters.gaussian(img, sigma=2, channel_axis=-1)` | `channel_axis=-1` indica que el último eje son los canales de color |
| **Grafo de conectividad 2D** | `grid_to_graph(n_x=H, n_y=W)` | Crea la matriz de adyacencia espacial entre píxeles vecinos para clustering |
| **Clustering Jerárquico** | `AgglomerativeClustering(n_clusters=10, linkage='ward', connectivity=conn).fit(X)` | `X` debe tener forma `(H*W, 1)` (`gray.reshape(-1, 1)`). Las etiquetas se recuperan con `.labels_.reshape(H, W)` |
| **Apilar frames de vídeo** | `stack = np.stack(frames, axis=0)` | Crea un array 4D `(N_frames, H, W, 3)` a partir de una lista de imágenes |
| **Fondo por mediana temporal** | `bg = np.median(stack, axis=0).astype(np.uint8)` | Calcula la mediana a lo largo del tiempo (`axis=0`) para eliminar objetos móviles |
| **Diferencia absoluta RGB** | `diff = np.abs(frame.astype(np.int16) - bg.astype(np.int16))` | Convertir a `int16` evita el desbordamiento (*underflow*) al restar `uint8` |
| **Distancia SSD (Ventana)** | `np.sum((window - template)**2)` o `np.linalg.norm(window - template)**2` | Mide **disimilitud** → el mejor match es el **mínimo** (`np.argmin` / `< threshold`) |
| **Correlación NCC** | `feature.match_template(img, template, pad_input=False)` | Mide **similitud** en `[-1, 1]` → el mejor match es el **máximo** (`np.argmax` / `> threshold`) |
| **Coordenadas `(y, x)` del pico**| `y, x = np.unravel_index(np.argmin(ssd), ssd.shape)` | En NCC se usa `np.argmax(ncc)`. Devuelve `(fila, columna)` |
| **Dibujar rectángulo en plot** | `ax.add_patch(plt.Rectangle((x, y), w, h, edgecolor='r', facecolor='none'))` | Recibe esquina superior izquierda en orden **`(x, y)` = `(col, fila)`**, `ancho`, `alto` |
| **Rotar plantilla** | `transform.rotate(template, angle, resize=False)` | `resize=True` amplía el lienzo para que no se recorten las esquinas al rotar |

<a id='2-segmentacion-skimage'></a>
---
## 2. Segmentación por Superpíxeles y Grafos: Felzenszwalb, SLIC y Quickshift

### 2.1 Comparativa de Parámetros de los Tres Algoritmos (`skimage.segmentation`)

1. **Felzenszwalb (`felzenszwalb(image, scale=100, sigma=0.5, min_size=50)`):**
   - **`scale`:** Controla el umbral de fusión del grafo. Valores **grandes** fusionan más regiones y producen **menos segmentos (más grandes)**; valores pequeños producen segmentos más finos y detallados.
   - **`sigma`:** Desviación estándar del filtro Gaussiano aplicado antes de segmentar. Mayor `sigma` reduce el ruido pero suaviza bordes.
   - **`min_size`:** Tamaño mínimo en píxeles permitido por región. Cualquier segmento menor se fusiona con sus vecinos, evitando la sobre-segmentación.

2. **SLIC (`slic(image, n_segments=250, compactness=10, sigma=1, start_label=1)`):**
   - Agrupa píxeles mediante **K-Means** combinando similitud de color (espacio CIE-Lab) y proximidad espacial $(x, y)$.
   - **`n_segments`:** Número aproximado de superpíxeles que se desea obtener.
   - **`compactness`:** Pondera la distancia espacial frente a la distancia de color. Un valor **alto** genera superpíxeles más regulares y cuadrados; un valor **bajo** permite que los bordes del segmento se adapten mejor a los contornos de la imagen.
   - **`sigma`:** Suavizado Gaussiano previo.

3. **Quickshift (`quickshift(image, kernel_size=3, max_dist=6, ratio=0.5)`):**
   - Algoritmo de búsqueda de modas (*mode-seeking*) sobre una estimación de densidad 5D (color + espacio).
   - **`kernel_size`:** Tamaño del kernel Gaussiano para estimar la densidad. Al **aumentar `kernel_size`** (`1 → 3 → 5 → 7`), la densidad se suaviza más y se obtienen **muchos menos segmentos (más grandes)**.
   - **`max_dist`:** Punto de corte en las distancias del árbol; valores mayores producen menos clusters.
   - **`ratio`:** Entre `0` y `1`, equilibra la importancia del color (`1`) frente a la posición espacial (`0`).

### 2.2 Preguntas Teóricas de Examen sobre Segmentación
- **¿Es necesario convertir la imagen a escala de grises para segmentar?**
  **No.** Felzenszwalb, SLIC y Quickshift funcionan directamente sobre imágenes **RGB**, y usar los 3 canales de color suele dar mejores resultados porque aprovecha las diferencias cromáticas para separar objetos.
- **¿Es necesario convertir la imagen a formato `float` `[0, 1]`?**
  **Sí, es recomendable** usar `img_as_float(image)` (especialmente después de redimensionar con `transform.resize`), ya que internamente los algoritmos esperan valores normalizados en `[0, 1]`.
- **¿Cómo afecta reducir la resolución de la imagen a los parámetros óptimos?**
  Al reducir el tamaño (ej. a `1/3` de alto y ancho), la imagen tiene 9 veces menos píxeles y menos detalle fino. Parámetros definidos en píxeles como `min_size` (en Felzenszwalb) o `kernel_size` (en Quickshift) pasan a cubrir una proporción mucho mayor de la imagen, por lo que deben reajustarse para evitar infra-segmentar o sobre-segmentar.
- **¿Cómo afecta el ruido Sal y Pimienta y cuál algoritmo es más robusto?**
  - El **ruido sal y pimienta** introduce píxeles extremos aislados que crean bordes irregulares y **disparan el número de segmentos en Felzenszwalb**.
  - Aplicar un **suavizado Gaussiano previo** (`sigma=2`) atenúa el ruido y devuelve segmentos mucho más limpios, aunque difumina ligeramente los bordes.
  - **SLIC es más robusto frente al ruido** que Felzenszwalb porque fuerza un número aproximado de regiones (`n_segments`) y penaliza la irregularidad espacial mediante `compactness`.

In [ ]:
# Plantilla general para aplicar y visualizar cualquier método de segmentación y contar segmentos
def aplicar_segmentacion(image, metodo='felzenszwalb', **params):
    img_fl = img_as_float(image[:, :, :3]) if image.ndim == 3 else img_as_float(image)

    if metodo == 'felzenszwalb':
        # Ej: scale=100, sigma=0.8, min_size=50
        segments = segmentation.felzenszwalb(img_fl, **params)
    elif metodo == 'slic':
        # Ej: n_segments=250, compactness=10, sigma=1, start_label=1
        segments = segmentation.slic(img_fl, **params)
    elif metodo == 'quickshift':
        # Ej: kernel_size=3, max_dist=6, ratio=0.5
        segments = segmentation.quickshift(img_fl, **params)

    num_segments = len(np.unique(segments))
    img_boundaries = segmentation.mark_boundaries(img_fl, segments)
    return segments, num_segments, img_boundaries

# Generación de versiones con ruido Sal y Pimienta y suavizado Gaussiano
def generar_variaciones_ruido(image, amount=0.05, sigma=2):
    salt_pepper = util.random_noise(image, mode='s&p', amount=amount)
    smoothed = filters.gaussian(salt_pepper, sigma=sigma, channel_axis=-1)
    return salt_pepper, smoothed

<a id='3-clustering-jerarquico'></a>
---
## 3. Segmentación mediante Clustering Jerárquico (`AgglomerativeClustering`)

Para segmentar una imagen en escala de grises usando **`AgglomerativeClustering`** de `scikit-learn` conservando la vecindad espacial entre píxeles:

1. **Convertir a escala de grises 2D:** `gray = color.rgb2gray(image)` de dimensiones `shape = (H, W)`.
2. **Aplanar a formato `(n_samples, n_features)`:** `X = gray.reshape(-1, 1)` convierte la matriz `(H, W)` en una columna de `H * W` filas y `1` característica (la intensidad del píxel).
3. **Crear matriz de conectividad espacial:** `connectivity = grid_to_graph(n_x=shape[0], n_y=shape[1])` indica qué píxeles son adyacentes en la rejilla 2D para que solo se puedan fusionar clusters contiguos.
4. **Ajustar el clustering con enlace de Ward:** `clustering = AgglomerativeClustering(n_clusters=10, linkage='ward', connectivity=connectivity).fit(X)` (el método de Ward minimiza la varianza dentro de cada cluster).
5. **Reconstruir la imagen 2D:** `labels = clustering.labels_.reshape(shape)` devuelve la matriz `(H, W)` con etiquetas de `0` a `n_clusters - 1`.

In [ ]:
# Plantilla completa de segmentación por Clustering Jerárquico (Ward + grid_to_graph)
def segmentar_clustering_jerarquico(image, n_clusters=10):
    gray_image = color.rgb2gray(image[:, :, :3]) if image.ndim == 3 else image
    shape = gray_image.shape

    # 1. Convertir a vector columna (n_pixeles, 1)
    X = gray_image.reshape(-1, 1)

    # 2. Grafo de conectividad espacial 2D
    connectivity = grid_to_graph(n_x=shape[0], n_y=shape[1])

    # 3. Clustering aglomerativo con criterio de Ward
    clustering = AgglomerativeClustering(
        n_clusters=n_clusters,
        linkage='ward',
        connectivity=connectivity
    )
    clustering.fit(X)

    # 4. Devolver etiquetas con la forma 2D original de la imagen
    labels = clustering.labels_.reshape(shape)
    return gray_image, labels

<a id='4-background-subtraction'></a>
---
## 4. Sustracción de Fondo para Detección de Movimiento (*Background Subtraction*)

El algoritmo de sustracción de fondo detecta objetos móviles en una secuencia de vídeo en dos fases:

### Fase 1: Estimación del Modelo de Fondo (`compute_background`)
- Se cargan los primeros $N$ fotogramas de entrenamiento (ej. 100 frames) y se redimensionan a la misma escala (ej. al 50% con `preserve_range=True` y `.astype(np.uint8)`).
- Se apilan en un único array 4D de dimensiones `(N, H, W, 3)` mediante `frame_stack = np.stack(frames, axis=0)`.
- Se calcula la **mediana a lo largo del eje temporal (`axis=0`)**:
  $$\text{Background}(y, x, c) = \text{median}_{t} \{ I_t(y, x, c) \}$$
  Como los objetos en movimiento solo ocupan un píxel durante unos pocos fotogramas, la mediana temporal recupera el color del fondo estático.

### Fase 2: Extracción de Objetos en Movimiento (`subtract_background`)
- Para cada fotograma de test (redimensionado al mismo tamaño que el fondo), se calcula la **diferencia absoluta** respecto al fondo:
  - **Importante:** Convertir ambos arrays a `np.int16` (o `float`) antes de restar (`test_frame.astype(np.int16) - background.astype(np.int16)`), porque en `uint8` una resta negativa produce *underflow* (ej. `10 - 20 = 246`).
- Para obtener una máscara binaria 2D `(H, W)` a partir de la diferencia en 3 canales `(H, W, 3)`, se toma el máximo de los 3 canales y se compara con un umbral `threshold` (ej. `30`):
  - `mask = np.max(difference, axis=2) > threshold` (`True` = objeto móvil / *foreground*, `False` = fondo estático).

In [ ]:
# Funciones completas para Sustracción de Fondo en secuencias de imágenes
def resize_frame(frame, scale=0.5):
    """Reduce las dimensiones del fotograma conservando el rango [0, 255] en uint8."""
    h, w = frame.shape[:2]
    return resize(
        frame,
        (int(h * scale), int(w * scale)),
        preserve_range=True,
        anti_aliasing=True
    ).astype(np.uint8)

def compute_background(frames):
    """Apila una lista de fotogramas y calcula la mediana en el eje temporal (axis=0)."""
    frame_stack = np.stack(frames, axis=0)          # Shape: (N, H, W, 3)
    background = np.median(frame_stack, axis=0)     # Shape: (H, W, 3)
    return background.astype(np.uint8)

def subtract_background(test_frame, background_image, threshold=30):
    """Calcula la diferencia absoluta entre frame y fondo y devuelve la máscara binaria 2D."""
    difference = np.abs(test_frame.astype(np.int16) - background_image.astype(np.int16))
    mask = np.max(difference, axis=2) > threshold   # Shape: (H, W) booleana
    return mask

<a id='5-template-matching'></a>
---
## 5. Template Matching: SSD vs Correlación Cruzada Normalizada (NCC)

El **Template Matching** busca la posición de una plantilla $T$ de tamaño `(h, w)` deslizando una ventana sobre una imagen fuente $I$ de tamaño `(H, W)`. El mapa de respuesta resultante tiene tamaño **`(H - h + 1, W - w + 1)`**.

### 5.1 Comparativa entre SSD y NCC
| Método | Qué mide | Cómo se calcula | Mejor Coincidencia (*Best Match*) | Condición de Umbral |
| :--- | :--- | :--- | :--- | :--- |
| **SSD** (*Sum of Squared Differences*) | **Distancia / Error** entre ventana y plantilla | $\sum (W - T)^2$ o `np.linalg.norm(W - T)**2` | **Valor MÍNIMO** (`np.argmin`) (`0` = idénticas) | `ssd_map < threshold` |
| **NCC** (*Normalized Cross-Correlation*) | **Similitud** normalizada en `[-1, 1]` | `feature.match_template(img, tpl, pad_input=False)` | **Valor MÁXIMO** (`np.argmax`) (`1` = idénticas) | `ncc_map > threshold` |

### 5.2 Dos Formas de Extraer las Coordenadas `(x, y)` para Dibujar el Rectángulo
- **Nota sobre coordenadas:** En NumPy los índices son `(fila, columna) = (y, x)`, pero `plt.Rectangle((x, y), w, h)` pide la esquina superior izquierda en orden **`(x, y) = (columna, fila)`** con ancho `w = template.shape[1]` y alto `h = template.shape[0]`.
1. **Búsqueda del único mejor punto con `np.unravel_index`:**
   - En SSD: `y_ssd, x_ssd = np.unravel_index(np.argmin(result_ssd), result_ssd.shape)`
   - En NCC: `y_ncc, x_ncc = np.unravel_index(np.argmax(result_ncc), result_ncc.shape)`
2. **Búsqueda de múltiples coincidencias mediante Componentes Conexas (`skimage.measure`):**
   - Tras binarizar el mapa con el umbral (`thresholded`), se etiquetan las regiones conexas con `labeled = measure.label(thresholded)` y se extrae el centroide de cada región con `measure.regionprops(labeled)`:
   - `locations = [(int(p.centroid[1]), int(p.centroid[0])) for p in measure.regionprops(labeled)]`.

In [ ]:
# Implementación de SSD, NCC y localización tanto por pico global como por componentes conexas
def ensure_gray_and_normalize(img):
    if img.ndim == 3:
        if img.shape[-1] == 4:
            img = color.rgba2rgb(img)
        img = color.rgb2gray(img)
    # Normalizar al rango [0.0, 1.0]
    return (img - np.min(img)) / (np.max(img) - np.min(img))

def ssd_match(image, template):
    """Calcula el mapa SSD deslizando la plantilla sobre la imagen."""
    h, w = template.shape
    H, W = image.shape
    ssd_map = np.zeros((H - h + 1, W - w + 1), dtype=np.float64)
    for y in range(H - h + 1):
        for x in range(W - w + 1):
            window = image[y:y + h, x:x + w]
            # Equivalente a: np.sum((window - template) ** 2)
            ssd_map[y, x] = np.linalg.norm(window - template) ** 2
    return ssd_map

def ncc_match(image, template):
    """Calcula el mapa NCC con match_template de skimage.feature."""
    return feature.match_template(image, template, pad_input=False)

def get_connected_component_centroids(binary_mask):
    """Extrae las coordenadas (x, y) de cada región activa en una máscara umbralizada."""
    labeled = measure.label(binary_mask)
    return [(int(prop.centroid[1]), int(prop.centroid[0])) for prop in measure.regionprops(labeled)]

<a id='6-rotacion-template'></a>
---
## 6. Invarianza a la Rotación en Template Matching (`transform.rotate`)

- **Rotación de la plantilla:**
  - `rotated_tpl = transform.rotate(template, angle, resize=True)` (o `resize=False` para mantener el tamaño original de la plantilla rellenando las esquinas con `0`).
- **¿Son SSD o NCC invariantes a la rotación? (Pregunta de Examen):**
  - **No, ninguno de los dos métodos es invariante a la rotación.** Ambos comparan píxel a píxel una rejilla rígida alineada con los ejes.
  - Con ángulos muy pequeños (**`2°` o `5°`**), la estructura espacial apenas cambia y ambos métodos (especialmente **NCC**, al estar normalizado frente a cambios locales de intensidad y bordes negros de la rotación) aún logran localizar el ojo.
  - A medida que el ángulo aumenta (**`10°`, `15°`, `20°`**), la correspondencia píxel a píxel se rompe: la distancia **SSD aumenta** por encima del umbral y la correlación **NCC disminuye** por debajo del umbral, provocando que **dejen de detectar el objeto** (si se usa un umbral fijo) o que el máximo/mínimo salte a una región errónea de la imagen.
  - Para lograr invarianza a la rotación con Template Matching clásico habría que rotar la plantilla en un banco de ángulos y quedarse con la mejor respuesta global, o bien emplear descriptores invariantes a rotación (como SIFT/ORB).

In [ ]:
# Plantilla para evaluar el efecto de rotar el template en distintos ángulos [2, 5, 10, 15, 20]
def evaluar_rotacion_template(image, template, angles=[2, 5, 10, 15, 20]):
    resultados = []
    for angle in angles:
        tpl_rot = transform.rotate(template, angle, resize=False)
        ssd_map = ssd_match(image, tpl_rot)
        ncc_map = ncc_match(image, tpl_rot)

        y_ssd, x_ssd = np.unravel_index(np.argmin(ssd_map), ssd_map.shape)
        y_ncc, x_ncc = np.unravel_index(np.argmax(ncc_map), ncc_map.shape)

        resultados.append({
            'angle': angle,
            'min_ssd': ssd_map[y_ssd, x_ssd], 'pos_ssd': (x_ssd, y_ssd),
            'max_ncc': ncc_map[y_ncc, x_ncc], 'pos_ncc': (x_ncc, y_ncc)
        })
    return resultados